# Receipt OCR — Experimentation Notebook

Step-by-step testing of each phase before integrating into the app.

**Phases:**
1. Load Kaggle dataset
2. OpenCV preprocessing
3. Google Vision OCR
4. Receipt parsing
5. SQLite database


---
## Phase 1 — Load Kaggle Dataset

In [ ]:
# Install dependencies
!pip install kagglehub opencv-python-headless pillow matplotlib -q

In [ ]:
# Option A: Upload kaggle.json manually (recommended for Colab)
from google.colab import files
import os

# Uncomment to upload your kaggle.json
# uploaded = files.upload()  # select your kaggle.json
# os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
# !cp kaggle.json ~/.kaggle/
# !chmod 600 ~/.kaggle/kaggle.json

print('Kaggle credentials ready.')

In [ ]:
import kagglehub

# Download receipt dataset
# This dataset contains real receipt images for testing
path = kagglehub.dataset_download('trainingdatapro/ocr-receipts-text-detection')
print(f'Dataset downloaded to: {path}')

In [ ]:
import os
from pathlib import Path

# List all image files in the dataset
image_extensions = {'.jpg', '.jpeg', '.png', '.bmp'}
image_files = []

for root, dirs, files in os.walk(path):
    for f in files:
        if Path(f).suffix.lower() in image_extensions:
            image_files.append(os.path.join(root, f))

print(f'Found {len(image_files)} images')
for f in image_files[:5]:
    print(' ', f)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

# Preview first 3 images
fig, axes = plt.subplots(1, min(3, len(image_files)), figsize=(15, 8))

for i, img_path in enumerate(image_files[:3]):
    img = mpimg.imread(img_path)
    axes[i].imshow(img, cmap='gray' if img.ndim == 2 else None)
    axes[i].set_title(f'Receipt {i+1}\n{Path(img_path).name}', fontsize=8)
    axes[i].axis('off')

plt.tight_layout()
plt.show()

print('Dataset loaded successfully!')

---
## Phase 2 — OpenCV Preprocessing

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

# Pick one image to experiment with
TEST_IMAGE_PATH = image_files[0]  # change index to try different receipts
print(f'Testing with: {TEST_IMAGE_PATH}')

# Load
original = cv2.imread(TEST_IMAGE_PATH)
print(f'Image shape: {original.shape}')

In [ ]:
def show_images(images_dict, cols=3, figsize=(18, 6)):
    """Helper to display multiple images side by side."""
    n = len(images_dict)
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(figsize[0], figsize[1] * rows))
    if rows == 1:
        axes = [axes] if cols == 1 else axes
    else:
        axes = axes.flatten()
    
    for idx, (title, img) in enumerate(images_dict.items()):
        cmap = 'gray' if img.ndim == 2 else None
        axes[idx].imshow(img if img.ndim == 2 else cv2.cvtColor(img, cv2.COLOR_BGR2RGB), cmap=cmap)
        axes[idx].set_title(title)
        axes[idx].axis('off')
    
    for idx in range(n, len(axes)):
        axes[idx].axis('off')
    
    plt.tight_layout()
    plt.show()


# Step-by-step preprocessing
gray      = cv2.cvtColor(original, cv2.COLOR_BGR2GRAY)
resized   = cv2.resize(gray, None, fx=1.5, fy=1.5, interpolation=cv2.INTER_CUBIC)
denoised  = cv2.fastNlMeansDenoising(resized, h=10)

# Sharpen
gaussian  = cv2.GaussianBlur(denoised, (0, 0), sigmaX=2)
sharpened = cv2.addWeighted(denoised, 1.5, gaussian, -0.5, 0)

# Threshold
thresholded = cv2.adaptiveThreshold(
    sharpened, 255,
    cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
    cv2.THRESH_BINARY, 15, 8
)

show_images({
    'Original': original,
    'Grayscale': gray,
    'Resized': resized,
    'Denoised': denoised,
    'Sharpened': sharpened,
    'Thresholded (final)': thresholded,
})

print('Preprocessing pipeline complete!')

---
## Phase 3 — Google Vision OCR

In [ ]:
!pip install google-cloud-vision -q

In [ ]:
# Upload your Google credentials JSON
from google.colab import files

# uploaded = files.upload()  # select google_credentials.json

import os
# os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = 'google_credentials.json'

print('Set GOOGLE_APPLICATION_CREDENTIALS env var above, then run next cell.')

In [ ]:
from google.cloud import vision

def run_ocr_on_image(cv_image: np.ndarray) -> str:
    """Send preprocessed CV image to Google Vision and return full text."""
    client = vision.ImageAnnotatorClient()
    
    _, buffer = cv2.imencode('.jpg', cv_image)
    content = buffer.tobytes()
    
    image = vision.Image(content=content)
    response = client.document_text_detection(image=image)
    
    if response.error.message:
        raise RuntimeError(f'Vision API error: {response.error.message}')
    
    return response.full_text_annotation.text


# Test OCR on preprocessed image
try:
    raw_text = run_ocr_on_image(thresholded)
    print('=== RAW OCR OUTPUT ===')
    print(raw_text)
except Exception as e:
    print(f'OCR failed: {e}')
    print('Make sure GOOGLE_APPLICATION_CREDENTIALS is set correctly.')

---
## Phase 4 — Receipt Parsing

In [ ]:
# Use mock OCR text if you don't have Vision API yet
MOCK_TEXT = """
INDOMARET
Jl. Pahlawan No.12, Jakarta
Tel: 021-5551234

28/06/2024  14:32:05

AQUA 600ML          2X   3.000    6.000
INDOMIE GORENG      3X   3.500   10.500
TEH BOTOL SOSRO     1X   5.000    5.000
POCARI SWEAT 500ML  2X   7.500   15.000
CHITATO BBQ         1X  12.000   12.000

------------------------------------
SUBTOTAL                    48.500
DISKON MEMBER                2.000
------------------------------------
TOTAL                       46.500

TUNAI                       50.000
KEMBALI                      3.500
"""

# Use this or replace with actual OCR output
text_to_parse = MOCK_TEXT  # or: text_to_parse = raw_text
print('Text to parse ready.')

In [ ]:
import re

def clean_number(s):
    return int(re.sub(r'[^\d]', '', s)) if re.sub(r'[^\d]', '', s) else 0

# Pattern: NAME   QTYx   PRICE   TOTAL
ITEM_RE = re.compile(r'^(.+?)\s+(\d+)[Xx]\s+([\d.,]+)\s+([\d.,]+)$')

items = []
for line in text_to_parse.splitlines():
    line = re.sub(r' {2,}', ' ', line).strip()
    m = ITEM_RE.match(line)
    if m:
        items.append({
            'item': m.group(1).strip().upper(),
            'qty': int(m.group(2)),
            'price': clean_number(m.group(3)),
            'total': clean_number(m.group(4)),
        })

import pandas as pd
df = pd.DataFrame(items)
print(df.to_string(index=False))
print(f'\nSum: Rp {df.total.sum():,}'.replace(',', '.'))

---
## Phase 5 — SQLite Database

In [ ]:
import sqlite3
from datetime import datetime

conn = sqlite3.connect('test_receipts.db')
conn.row_factory = sqlite3.Row
conn.execute('PRAGMA foreign_keys = ON')

conn.executescript("""
CREATE TABLE IF NOT EXISTS receipts (
    id           INTEGER PRIMARY KEY AUTOINCREMENT,
    store_name   TEXT,
    receipt_date TEXT,
    total        INTEGER,
    raw_text     TEXT,
    created_at   TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);

CREATE TABLE IF NOT EXISTS items (
    id         INTEGER PRIMARY KEY AUTOINCREMENT,
    receipt_id INTEGER NOT NULL,
    item_name  TEXT,
    qty        INTEGER,
    price      INTEGER,
    total      INTEGER,
    FOREIGN KEY (receipt_id) REFERENCES receipts(id) ON DELETE CASCADE
);
""")

conn.commit()
print('Database ready.')

In [ ]:
# Insert parsed receipt
cur = conn.execute(
    'INSERT INTO receipts (store_name, receipt_date, total, raw_text) VALUES (?,?,?,?)',
    ('INDOMARET', '28/06/2024', 46500, text_to_parse)
)
receipt_id = cur.lastrowid

for item in items:
    conn.execute(
        'INSERT INTO items (receipt_id, item_name, qty, price, total) VALUES (?,?,?,?,?)',
        (receipt_id, item['item'], item['qty'], item['price'], item['total'])
    )

conn.commit()
print(f'Inserted receipt #{receipt_id} with {len(items)} items.')

In [ ]:
# Query everything back
rows = conn.execute(
    'SELECT i.*, r.store_name FROM items i JOIN receipts r ON r.id = i.receipt_id'
).fetchall()

result_df = pd.DataFrame([dict(r) for r in rows])
print(result_df.to_string(index=False))

conn.close()